# Hayato Dashboard Parquet PoC

ColabでDuckDBを使い、Google Driveの `MyDrive/ハヤトの野望/` にPoC用Parquetを生成する。

In [ ]:
!pip -q install duckdb pandas


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

OUTPUT_DIR = Path('/content/drive/MyDrive/ハヤトの野望')
OUTPUT_PATH = OUTPUT_DIR / 'hayato_live_poc.parquet'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROW_COUNT = 10_000

base_rows = [
    {
        'video_id': 'demo_live_001',
        'title': 'なんでもできる超リアルな都市で好き放題してみた',
        'published_at': '2026-04-30T12:00:00+09:00',
        'duration_sec': 11630,
        'live_type': '通常LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 2688,
        'thumbnail_url': 'https://i.ytimg.com/vi/dQw4w9WgXcQ/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_002',
        'title': 'ハヤトの野望作戦会議',
        'published_at': '2026-04-28T12:00:00+09:00',
        'duration_sec': 7579,
        'live_type': '通常LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 1284,
        'thumbnail_url': 'https://i.ytimg.com/vi/aqz-KE-bpKQ/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_003',
        'title': 'shapez2 無限立体構造',
        'published_at': '2026-04-24T12:00:00+09:00',
        'duration_sec': 12941,
        'live_type': '通常LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 2513,
        'thumbnail_url': 'https://i.ytimg.com/vi/M7lc1UVf-VE/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_004',
        'title': '戦国武将になって天下統一を目指す #2',
        'published_at': '2026-04-17T12:00:00+09:00',
        'duration_sec': 11912,
        'live_type': '通常LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 2816,
        'thumbnail_url': 'https://i.ytimg.com/vi/dQw4w9WgXcQ/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_005',
        'title': '戦国武将になって天下統一を目指す',
        'published_at': '2026-04-16T12:00:00+09:00',
        'duration_sec': 12344,
        'live_type': '通常LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 3365,
        'thumbnail_url': 'https://i.ytimg.com/vi/aqz-KE-bpKQ/mqdefault.jpg',
    },
]

rows = []
for i in range(ROW_COUNT):
    base = base_rows[i % len(base_rows)].copy()
    base['video_id'] = f"{base['video_id']}_{i + 1:05d}"
    base['title'] = f"{base['title']} #{i + 1:05d}"
    base['published_at'] = (
        pd.Timestamp('2026-04-30T12:00:00+09:00') - pd.Timedelta(days=i % 365)
    ).isoformat()
    base['duration_sec'] = int(base['duration_sec'] + (i % 1800))
    base['peak_concurrent_viewers'] = int(base['peak_concurrent_viewers'] + ((i * 37) % 5000))
    rows.append(base)

df = pd.DataFrame(rows)
df['published_at'] = pd.to_datetime(df['published_at'])

con = duckdb.connect()
con.register('videos_df', df)
con.execute(
    f"""
    COPY (
      SELECT
        video_id,
        title,
        published_at,
        duration_sec,
        live_type,
        visibility,
        peak_concurrent_viewers,
        thumbnail_url
      FROM videos_df
      ORDER BY peak_concurrent_viewers DESC
    ) TO '{OUTPUT_PATH.as_posix()}' (
      FORMAT PARQUET,
      COMPRESSION ZSTD
    )
    """
)

print(f'rows: {len(df)}')
print(f'created: {OUTPUT_PATH}')
print(f'size: {OUTPUT_PATH.stat().st_size} bytes')


In [ ]:
con.execute(f"DESCRIBE SELECT * FROM read_parquet('{OUTPUT_PATH.as_posix()}')").df()


In [ ]:
con.execute(
    f"""
    SELECT
      video_id,
      title,
      published_at,
      live_type,
      visibility,
      peak_concurrent_viewers
    FROM read_parquet('{OUTPUT_PATH.as_posix()}')
    ORDER BY peak_concurrent_viewers DESC
    LIMIT 20
    """
).df()
